In [ ]:
DATA_FILE = Path(r"open-meteo-data.csv")

if not DATA_FILE.is_file():
    matches = list(Path(".").rglob(DATA_FILE.name))
    if not matches:
        raise FileNotFoundError(f"Dataset not found: {DATA_FILE.name}")
    DATA_FILE = matches[0]

DATA_DIR = DATA_FILE.parent
files = [DATA_FILE]

print(f"Using dataset: {DATA_FILE.resolve()}")
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import warnings
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# Cell 0 — Dataset discovery and initial inspection


warnings.filterwarnings("ignore")

DATA_DIR = Path("open-meteo-raw-data.csv")
SUPPORTED_EXTENSIONS = {".csv", ".txt", ".xlsx", ".xls", ".parquet", ".json"}

files = [
    p for p in DATA_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
]

'''
if not files:
    try:
        print("No datasets found in /content. Please upload your files.")
        uploaded = colab_files.upload()
        files = [
            DATA_DIR / filename
            for filename in uploaded.keys()
        ]
    except ImportError:
        print("No supported datasets found in /content.")
'''

print(f"\nFound {len(files)} dataset file(s):")
for file_path in files:
    print(f" - {file_path} ({file_path.stat().st_size / 1024**2:.2f} MB)")


def load_dataset(path):
    """Load common tabular file formats."""
    suffix = path.suffix.lower()

    if suffix in {".csv", ".txt"}:
        try:
            return pd.read_csv(path, low_memory=False)
        except Exception:
            return pd.read_csv(path, sep=None, engine="python", low_memory=False)

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    if suffix == ".json":
        return pd.read_json(path)

    raise ValueError(f"Unsupported file format: {path}")


def find_timestamp_columns(df):
    candidates = []
    keywords = ["time", "date", "datetime", "timestamp", "valid", "issue"]

    for col in df.columns:
        name = str(col).lower()
        if any(keyword in name for keyword in keywords):
            candidates.append(col)

    return candidates


def inspect_dataset(path):
    print("\n" + "=" * 100)
    print(f"FILE: {path.name}")
    print("=" * 100)
    if path.name != "open-meteo-raw-data.csv":
        return None

    print("\nLoading open-meteo-raw-data.csv directly...")
    df = pd.read_csv(DATA_DIR / "open-meteo-raw-data.csv")

    # Parse datetime columns and create time-based features.
    timestamp_columns = find_timestamp_columns(df)
    for column in timestamp_columns:
        parsed = pd.to_datetime(df[column], errors="coerce")
        if parsed.notna().sum() > 0:
            df[column] = parsed
            df["year"] = parsed.dt.year
            df["month"] = parsed.dt.month
            df["day"] = parsed.dt.day
            df["hour"] = parsed.dt.hour
            df["dayofweek"] = parsed.dt.dayofweek

    # Remove completely empty columns and duplicate rows.
    df = df.dropna(axis=1, how="all").drop_duplicates()

    # Fill missing numeric values using interpolation, then forward/backward fill.
    numeric_columns = df.select_dtypes(include=np.number).columns
    if len(numeric_columns):
        df[numeric_columns] = (
            df[numeric_columns]
            .interpolate(limit_direction="both")
            .ffill()
            .bfill()
        )

    # Fill missing categorical values with the most frequent value.
    categorical_columns = df.select_dtypes(exclude=np.number).columns
    for column in categorical_columns:
        if df[column].isna().any():
            mode = df[column].mode()
            df[column] = df[column].fillna(mode.iloc[0] if len(mode) else "unknown")

    # Select a weather variable as the prediction target.
    preferred_targets = [
        "temperature_2m",
        "temperature_2m_mean",
        "temperature",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]
    target_column = next(
        (column for column in preferred_targets if column in df.columns),
        None,
    )

    if target_column is None:
        numeric_columns = df.select_dtypes(include=np.number).columns
        if not len(numeric_columns):
            raise ValueError("No numeric column is available for model training.")
        target_column = numeric_columns[-1]

    print(f"\nTraining target: {target_column}")

    model_data = df.copy()

    # Convert datetime columns to numeric timestamps.
    for column in model_data.select_dtypes(include=["datetime", "datetimetz"]).columns:
        model_data[column] = model_data[column].astype("int64") // 10**9

    X = model_data.drop(columns=[target_column])
    y = pd.to_numeric(model_data[target_column], errors="coerce")

    valid_rows = y.notna()
    X = X.loc[valid_rows]
    y = y.loc[valid_rows]

    # Encode categorical columns and ensure all features are numeric.
    X = pd.get_dummies(X, drop_first=False)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    split_index = max(1, int(len(X) * 0.8))
    X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
    y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]


    model = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    )

    model.fit(X_train, y_train)

    if len(X_test):
        predictions = model.predict(X_test)
        print("Test MAE:", mean_absolute_error(y_test, predictions))
        print("Test RMSE:", mean_squared_error(y_test, predictions) ** 0.5)

    CLEANED_DATA = df
    XGBOOST_MODEL = model
    MODEL_FEATURES = X.columns.tolist()

    print(f"Cleaned dataset shape: {df.shape}")
    print("XGBoost model trained successfully.")
DATASETS = {}

for file_path in files:
    dataset = inspect_dataset(file_path)
    if dataset is not None:
        DATASETS[file_path.name] = dataset

print("\nInspection complete.")
print("Loaded dataset variables:")
for name, dataset in DATASETS.items():
    print(f" - DATASETS[{name!r}]: {dataset.shape}")

Using dataset: C:\Users\nitin\Desktop\synapse making\notebooks\open-meteo-raw-data.csv

Found 0 dataset file(s):

Inspection complete.
Loaded dataset variables:


In [19]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import csv

warnings.filterwarnings("ignore")

DATA_FILE = Path(
    r"C:\Users\nitin\Desktop\synapse making\notebooks\open-meteo-raw-data.csv"
)

if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_FILE}")

print(f"Using dataset: {DATA_FILE}")
df = pd.read_csv(
    DATA_FILE,
    skiprows=header_index,
    header=0,
    engine="python",
    on_bad_lines="warn",
)

df = df.dropna(axis=1, how="all")
# Open-Meteo CSV files may contain metadata rows before the actual table.
with DATA_FILE.open("r", encoding="utf-8-sig", newline="") as file:
    rows = list(csv.reader(file))

if not rows:
    raise ValueError("The CSV file is empty.")

# The actual header is usually the first row with the largest number of fields.
header_index = max(
    range(len(rows)),
    key=lambda index: len(rows[index]),
)

expected_fields = len(rows[header_index])

if expected_fields < 2:
    raise ValueError("Could not identify a valid tabular header.")

df = pd.read_csv(
    DATA_FILE,
    skiprows=header_index,
    header=0,
    engine="python",
    on_bad_lines="warn",
)

# Remove completely empty columns created by malformed trailing fields.
df = df.dropna(axis=1, how="all")

print(f"Loaded CSV using header row {header_index + 1}.")

print("Original shape:", df.shape)
display(df.head())


def find_timestamp_columns(dataframe):
    keywords = ["time", "date", "datetime", "timestamp", "valid", "issue"]
    return [
        column
        for column in dataframe.columns
        if any(keyword in str(column).lower() for keyword in keywords)
    ]


# Parse timestamps and create time features
for column in find_timestamp_columns(df):
    parsed = pd.to_datetime(df[column], errors="coerce")

    if parsed.notna().any():
        df[column] = parsed
        df["year"] = parsed.dt.year
        df["month"] = parsed.dt.month
        df["day"] = parsed.dt.day
        df["hour"] = parsed.dt.hour
        df["dayofweek"] = parsed.dt.dayofweek


# Remove empty columns and duplicate rows
df = df.dropna(axis=1, how="all").drop_duplicates()

# Fill missing numeric values
numeric_columns = df.select_dtypes(include=np.number).columns

if len(numeric_columns):
    df[numeric_columns] = (
        df[numeric_columns]
        .interpolate(limit_direction="both")
        .ffill()
        .bfill()
    )

# Fill missing text values
categorical_columns = df.select_dtypes(exclude=np.number).columns

for column in categorical_columns:
    if df[column].isna().any():
        mode = df[column].mode()
        df[column] = df[column].fillna(
            mode.iloc[0] if len(mode) else "unknown"
        )


# Select prediction target
preferred_targets = [
    "temperature_2m",
    "temperature_2m_mean",
    "temperature",
    "relative_humidity_2m",
    "wind_speed_10m",
]

target_column = next(
    (column for column in preferred_targets if column in df.columns),
    None,
)

if target_column is None:
    numeric_columns = df.select_dtypes(include=np.number).columns

    if not len(numeric_columns):
        raise ValueError("No numeric column is available for model training.")

    target_column = numeric_columns[-1]

print(f"Training target: {target_column}")


# Prepare model data
model_data = df.copy()

for column in model_data.select_dtypes(
    include=["datetime", "datetimetz"]
).columns:
    model_data[column] = model_data[column].astype("int64") // 10**9

X = model_data.drop(columns=[target_column])
y = pd.to_numeric(model_data[target_column], errors="coerce")

valid_rows = y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

X = pd.get_dummies(X, drop_first=False)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

split_index = int(len(X) * 0.8)

if split_index == 0:
    raise ValueError("Not enough rows available for training.")

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]


# Train model
model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
)

model.fit(X_train, y_train)

if len(X_test):
    predictions = model.predict(X_test)
    print("Test MAE:", mean_absolute_error(y_test, predictions))
    print("Test RMSE:", mean_squared_error(y_test, predictions) ** 0.5)

CLEANED_DATA = df
XGBOOST_MODEL = model
MODEL_FEATURES = X.columns.tolist()

CLEANED_DATA.to_csv("open-meteo-cleaned.csv", index=False)

print(f"Cleaned dataset shape: {CLEANED_DATA.shape}")
print("Saved: open-meteo-cleaned.csv")
print("XGBoost model trained successfully.")

Using dataset: C:\Users\nitin\Desktop\synapse making\notebooks\open-meteo-raw-data.csv
Loaded CSV using header row 4.
Original shape: (26280, 11)


,time,temperature_2m (°C),relative_humidity_2m (%),dew_point_2m (°C),cloud_cover (%),wind_speed_10m (km/h),wind_direction_10m (°),shortwave_radiation (W/m²),direct_radiation (W/m²),diffuse_radiation (W/m²),direct_normal_irradiance (W/m²)
0,2017-01-01T00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-01-01T01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-01-01T02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-01-01T03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-01-01T04:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Training target: dayofweek
Test MAE: 1.9468748569488525
Test RMSE: 2.321672753706942
Cleaned dataset shape: (26280, 16)
Saved: open-meteo-cleaned.csv
XGBoost model trained successfully.
